# Conditional Edge

## 조건부 Edge(`Conditional Edge`) 사용해 분기 만들기
* `Conditional Edge`를 이용해 상황에 따라 다른 노드로 연결되도록 구현할 수 있습니다.

In [ ]:
from typing import Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph.message import add_messages  #Annotated라는 문법을 쓰기 위해 가져옴. 기존 내용에 메세지 추가

from langchain_core.messages import BaseMessage, HumanMessage

class AgentState(BaseModel):
    # Annotated[타입, 메타데이터] -> 해당 변수에 '메타데이터'에 해당하는 특성(기능)을 부여함
    # add_messages -> message에 적용하는 메타데이터, Node에서 history에 추가할 항목만 반환해도 알아서 리스트에 추가해 줌
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list) # 맨 처음 생성 시 빈 리스트로 시작
    last_player: Literal['A', 'B']
    number: int
    end: bool

In [10]:
def node_A(state: AgentState) -> dict:
    num = state.number
    new_num = num * 2
    return {
        'chat_history': [HumanMessage(name='A', content=f"숫자를 {num}에서 {new_num}(으)로 바꿈")],
        'number': new_num,
        'last_player': 'A'
    }

In [3]:
def node_B(state: AgentState) -> dict:
    num = state.number
    new_num = num - 1
    return {
        'chat_history': [HumanMessage(name='B', content=f"숫자를 {num}에서 {new_num}(으)로 변경")],
        'number': new_num,
        'last_player': 'B'
    }

In [ ]:
def node_judge(state: AgentState) -> dict:
    end = True if state.number > 100 else False
    return {
        'chat_history': [HumanMessage(name='judge', content=f"최종 숫자 {state.number}(으)로 종료합니다" if end else "계속 진행하세요")],
        # end가 True면 if 앞에 있는 문장이 실행되고, False면 else 뒤에 있는 문장이 실행됨
        'end': end
    }

In [11]:
from langgraph.graph import START, END, StateGraph

workflow = StateGraph(AgentState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('judge', node_judge)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'judge')
workflow.add_edge('B', 'judge')

## `route` 만들고 `conditional edge`에 부여하기
* `route`는 조건에 따라서 달라지는 목적지 `node`명 을 반환하는 함수입니다
* `workflow.add_conditional_edges`함수에 `route`를 적용하면, route가 반환한 `node`명으로 라우팅됩니다.

In [12]:
def judge_route(state: AgentState):
    # end가 True면 END로,
    # 마지막 플레이어가 B면 A로, A면 B로 가는 route
    if state.end:
        return END
    elif state.last_player == 'B':
        return 'A'
    else:
        return 'B'

workflow.add_conditional_edges(
    'judge', 
    judge_route
)

In [14]:
app = workflow.compile()

init_input = AgentState(chat_history=[], last_player='A', number=10, end=False)

for chunk in app.stream(init_input):
    print(chunk)

{'A': {'chat_history': [HumanMessage(content='숫자를 10에서 20(으)로 바꿈', additional_kwargs={}, response_metadata={}, name='A', id='caf4df91-4bfa-4dad-a6ad-fcb2f409a207')], 'number': 20, 'last_player': 'A'}}
{'judge': {'chat_history': [HumanMessage(content='계속 진행하세요', additional_kwargs={}, response_metadata={}, name='judge', id='3194b918-99c6-49ba-96b4-44e9162f3d6c')], 'end': False}}
{'B': {'chat_history': [HumanMessage(content='숫자를 20에서 19(으)로 변경', additional_kwargs={}, response_metadata={}, name='B', id='7316b373-f419-427a-9993-ee64f4e298ba')], 'number': 19, 'last_player': 'B'}}
{'judge': {'chat_history': [HumanMessage(content='계속 진행하세요', additional_kwargs={}, response_metadata={}, name='judge', id='a5f36ba5-47ff-41ad-9e56-f7bb39f3c0b4')], 'end': False}}
{'A': {'chat_history': [HumanMessage(content='숫자를 19에서 38(으)로 바꿈', additional_kwargs={}, response_metadata={}, name='A', id='6fd3c74f-a3d0-4513-9de1-d413ed7facd7')], 'number': 38, 'last_player': 'A'}}
{'judge': {'chat_history': [HumanMessag

## 토마토 게임 만들기
* ai와 토마토 게임을 진행할 수 있는 LangGraph 기반 게임을 만들어 봅시다
* 규칙: 두 명이 번갈아가면서 '토마토'를 한글자씩 말합니다. 올바르게 말하지 못하면 패배합니다.

In [ ]:
#1. State 정의하기

from pydantic import BaseModel
from typing import Literal, Optional

TOMATO = "토마토"  # 토(0) → 마(1) → 토(2) → 토 → 마 → 토 ...

class TomatoGameState(BaseModel):
    turn: int = 0                              # 지금까지 맞게 말한 글자 수 (다음 정답 위치 = turn % 3)
    player_input: str = ''                     # 방금 말한 글자
    last_player: Literal['user', 'ai'] = 'user'
    end: bool = False
    message: str = ''

In [ ]:
#2. Node 함수 정의하기

import random

def user_node(state: TomatoGameState) -> dict:
    expected = TOMATO[state.turn % len(TOMATO)]
    user_input = input(f"당신의 대답 (힌트: {expected}): ").strip()
    return {
        'player_input': user_input,
        'last_player': 'user'  #다음 분기에서 AI로 넘어가게됨
    }

def ai_node(state: TomatoGameState) -> dict:
    answer = random.choices(['토', '마'], weights=[2, 1])[0]  # 토 2/3, 마 1/3
    # print(f"AI의 대답: {answer}")
    return {
        'player_input': answer,
        'last_player': 'ai'
    }

def judge_node(state: TomatoGameState) -> dict:
    expected = TOMATO[state.turn % len(TOMATO)]

    if state.player_input != expected:
        #틀리면 게임 종료
        if state.last_player == 'user':
            msg = '< user 패배! >'
        else:
            msg = '< AI 패배! >'
        # print(msg)
        return {
            'end': True,
            'message': msg
        }
    #맞으면 턴 증가, 게임 계속됨
    return {
        'turn': state.turn + 1,
        'end': False,
        'message': f"'{state.player_input}'"
    }

In [31]:
#3. 그래프 연결하기

from langgraph.graph import START, END, StateGraph

workflow = StateGraph(TomatoGameState)

workflow.add_node('user', user_node)
workflow.add_node('ai', ai_node)
workflow.add_node('judge', judge_node)

workflow.add_edge(START, 'user')
workflow.add_edge('user', 'judge')
workflow.add_edge('ai', 'judge')

In [33]:
#4. judge_route+conditional edges

def judge_route(state: TomatoGameState):
    # end가 True면 END로,
    # 마지막 플레이어가 user면 ai로, ai면 user로 가는 route
    if state.end:
        return END
    elif state.last_player == 'user':
        return 'ai'
    else:
        return 'user'

workflow.add_conditional_edges(
    'judge',
    judge_route
)

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


In [34]:
app = workflow.compile()

init_input = TomatoGameState()

print("토마토게임 시~작!")
for chunk in app.stream(init_input):
    for node_name, update in chunk.items():
        if node_name == 'user' and 'player_input' in update:
            print(f"user:{update['player_input']}")
        elif node_name == 'ai' and 'player_input' in update:
            print(f"AI:{update['player_input']}")
        elif node_name == 'judge' and update.get('end'):
            print(update['message'])  # user 패배! / AI 패배!

토마토게임 시~작!
user:토
AI:마
user:토
AI:토
user:마
AI:토
user:마
< user 패배! >


### 팬아웃(Fan-Out) 구현하기
* 하나의 노드에서 여러 노드로 갈라지고(팬아웃) 다시 한 노드로 흐름을 모을 (팬인) 수 있습니다

In [35]:
from pydantic import BaseModel
from typing import Optional, Any

class FanOutState(BaseModel):
    value1: Optional[Any] = None
    value2: Optional[Any] = None

In [36]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"B 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value2 + 1}

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')  #한 노드에서 양갈래로 갈라지게(팬 아웃) 하면 병렬 처리가 된다!

app = workflow.compile()

In [38]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)
C 노드 실행 중... (2/3)
B 노드 실행 중... (3/3)C 노드 실행 중... (3/3)



{'value1': 1, 'value2': 1}

## 팬인(Fan-In) 구현하기
* `add_edge` 함수의 출발점은 여러 노드의 배열도 입력받을 수 있습니다.

In [39]:
def node_D(state: FanOutState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value1 + 1}

In [40]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')  #B,C에서 D로 팬 인 되면 다음으로 넘어간다.
workflow.add_edge('D', END)

app = workflow.compile()

In [41]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)
C 노드 실행 중... (2/3)
C 노드 실행 중... (3/3)B 노드 실행 중... (3/3)

D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 1, 'value2': 2}

### Q. 팬인 시 노드 종료 시점이 다르다면?

In [60]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

# 두 노드의 실행 시간을 다르게 하고 결과를 확인해보세요.
def node_B(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"B 노드 실행 중... ({i}/3)")
        time.sleep(2)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict:
    for i in range(1, 3):
        print(f"C 노드 실행 중... ({i}/2)")
        time.sleep(3)
    return {"value2": state.value2 + 1}

In [61]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [62]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

#B, C 모두 정해진 회차씩 돌고 올때까지 D로 넘어가지 않고 기다린다.

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/2)
B 노드 실행 중... (2/3)
C 노드 실행 중... (2/2)
B 노드 실행 중... (3/3)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 1, 'value2': 2}

# `Reducer` 활용하기

In [ ]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 6):
        print(f"B 노드 실행 중... ({i}/5)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}   

# node 두개가 동일한 value1이라는 변수를 바꾸려고 하는 상황(충돌)

In [64]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [ ]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

#InvalidUpdateError: At key 'value1': Can receive only one value per step. Use an Annotated key to handle multiple values.

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)
B 노드 실행 중... (3/5)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)


InvalidUpdateError: At key 'value1': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE

In [ ]:
from typing import Annotated

def numberAddReducer(left: int, right: int) -> int: # left: 기존 State, right: return 받은 state
    return left + right # return 받은 값으로 대체하지 않고, 기존 값에 return 받은 값을 더함

class ReducerState(BaseModel):
    value1: Annotated[Optional[Any], numberAddReducer] = None  # value1에 numberAddReducer 특성 붙여주기.
    value2: Optional[Any] = None

In [72]:
def node_B(state: ReducerState) -> dict: # state 정의도 모두 변경
    for i in range(1, 6):
        print(f"B 노드 실행 중... ({i}/5)")
        time.sleep(1)
    return {"value1": 1} # 새로 더해줄 값만 반환하면 됨

def node_C(state: ReducerState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": 1}

def node_D(state: ReducerState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value1 + 1}


In [73]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [74]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)
B 노드 실행 중... (3/5)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 2, 'value2': 3}

### 동적 팬아웃 구현하기
* `Conditional Edge`를 이용하면 Fan-Out을 동적으로 설정할 수 있습니다
* Fan-Out을 설정하려면, 목적지 노드 이름을 리스트로 모두 전달하면 됩니다.

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')

def fanout_router(state: ReducerState):
    if state.value2 > 0:
        return 'B'
    else:
        return ['B', 'C']
    
workflow.add_conditional_edges('A', fanout_router)

workflow.add_edge(['B', 'C'], 'D')

def end_router(state: ReducerState):
    if state.value1 > 2:
        return END
    else:
        return 'A'

workflow.add_conditional_edges('D', end_router)  #목적지 노드를 리스트로 전달


app = workflow.compile()

In [76]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)B 노드 실행 중... (3/5)

B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)
A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
B 노드 실행 중... (2/5)
B 노드 실행 중... (3/5)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)


{'value1': 3, 'value2': 3}

### 라우팅 심화 문제
* Fan-Out과 Reducing, 조건부 Edge를 적극적으로 활용해 음식점 운영을 최적화해봅시다 (괄호 안의 숫자는 소요시간(초))
```
burger: patty(5), bun(3) 각각 필요 -> burgerSet(2)
fries: fry(3) -> friesSet(1)
beverage: beverage(2)

세 가지가 모두 준비되면 종료
```

[프롬프트]
햄버거(패티, 번 각각 필요), 감자튀김, 음료수를 각각 만들어 메뉴가 완성되면 완료 메세지를 출력하는 코드를 짜려고 해.
각 메뉴를 만드는데에 드는 시간 지연은 아래쪽에 node들을 정의하는 cell 안에 적어두었어. 
1. 입력(menu_ordered)은 burger, fries, beverage 세가지 중 일부 또는 전부를 리스트로 받을 수 있어.
2. 입력받은 메뉴에 대해서 그 메뉴를 준비 시작하는 메세지를 출력하는 노드로 가야해.
3. 그리고 조리 시작하는 노드로 가야해. 조리가 완료되면 세트를 만들기 시작해.
4. 각 메뉴가 완료되면 menu_complete에 적절히 업데이트하고, 주문받은 모든 메뉴가 완료되면 "주문하신 버거, 감자튀김, 음료 나왔습니다." 이런식으로 주문한 메뉴를 언급하고 마치고 싶어.
5. 병렬처리 지침: 음식 준비는 동시에 시작할 수 있어. 버거 준비는 준비 시작 후 패티와 번을 동시에 만들 수 있지만, 둘다 만들어 진 후에야 버거 세트 조립을 시작할 수 있어.
참고 코드는 맨 아래 두개 cell에 적혀있어. 부족한 정보가 있다면 추가로 질문해줘.

In [77]:
# menuReducer는 None 처리를 넣는 편이 안전합니다. g_check_done이 여러 번 호출될 수 있어 중복 출력 방지용 served도 추가합니다.

from pydantic import BaseModel, Field
from typing import List, Annotated, Literal, Optional

def menuReducer(left: Optional[List[str]], right: Optional[List[str]]) -> List[str]:
    if left is None:
        left = []
    if right is None:
        right = []
    return left + right

class RestaurantState(BaseModel):
    menu_ordered: List[Literal['burger', 'fries', 'beverage']]
    menu_complete: Annotated[List[Literal['burger', 'fries', 'beverage']], menuReducer] = Field(default_factory=list)
    served: bool = False  # 완료 메시지 중복 출력 방지

In [84]:
import time

# 각종 그래프 노드 실행 및 상태 갱신 통합: 반환값으로 state의 menu_complete를 적절히 업데이트하도록 구성
def g_burger_start(state: RestaurantState) -> dict:
    print("버거 준비 시작")
    return {}

def g_fries_start(state: RestaurantState) -> dict:
    print("감자튀김 준비 시작")
    return {}

def g_patty(state: RestaurantState) -> dict:
    print("패티 조리 시작")
    time.sleep(5)
    print("패티 조리 완료")
    return {}

def g_bun(state: RestaurantState) -> dict:
    print("번 조리 시작")
    time.sleep(3)
    print("번 조리 완료")
    return {}

def g_burger_set(state: RestaurantState) -> dict:
    print("버거 세트 조립 시작")
    time.sleep(2)
    print("버거 세트 조립 완료")
    return {}

def g_burger_complete(state: RestaurantState) -> dict:
    print("버거 완성")
    return {'menu_complete': ['burger']}

def g_fry(state: RestaurantState) -> dict:
    print("감자튀김 조리 시작")
    time.sleep(3)
    print("감자튀김 조리 완료")
    return {}

def g_fries_set(state: RestaurantState) -> dict:
    print("감자튀김 세트 준비 시작")
    time.sleep(1)
    print("감자튀김 세트 준비 완료")
    return {}

def g_fries_complete(state: RestaurantState) -> dict:
    print("감자튀김 완성")
    return {'menu_complete': ['fries']}

def g_beverage(state: RestaurantState) -> dict:
    print("음료 준비 시작")
    time.sleep(2)
    print("음료 준비 완료")
    return {'menu_complete': ['beverage']}


#g_check_done 추가
MENU_LABEL = {
    'burger': '버거',
    'fries': '감자튀김',
    'beverage': '음료',
}

def g_check_done(state: RestaurantState) -> dict:
    ordered = set(state.menu_ordered)
    complete = set(state.menu_complete)

    if ordered <= complete and not state.served:
        # 주문 순서대로 한글 이름 나열
        labels = [MENU_LABEL[m] for m in state.menu_ordered]
        print(f"주문하신 {', '.join(labels)} 나왔습니다")
        return {'served': True}
    return {}

In [85]:
#앞에서 배운 동적 fan-out + fan-in 패턴을 그대로 씁니다.

from langgraph.graph import StateGraph, START, END

workflow = StateGraph(RestaurantState)

# 노드 등록
workflow.add_node('g_burger_start', g_burger_start)
workflow.add_node('g_fries_start', g_fries_start)
workflow.add_node('g_patty', g_patty)
workflow.add_node('g_bun', g_bun)
workflow.add_node('g_burger_set', g_burger_set)
workflow.add_node('g_burger_complete', g_burger_complete)
workflow.add_node('g_fry', g_fry)
workflow.add_node('g_fries_set', g_fries_set)
workflow.add_node('g_fries_complete', g_fries_complete)
workflow.add_node('g_beverage', g_beverage)
workflow.add_node('g_check_done', g_check_done)

# 1) START → 주문한 메뉴의 시작 노드로 동적 fan-out
def start_router(state: RestaurantState):
    targets = []
    if 'burger' in state.menu_ordered:
        targets.append('g_burger_start')
    if 'fries' in state.menu_ordered:
        targets.append('g_fries_start')
    if 'beverage' in state.menu_ordered:
        targets.append('g_beverage')

    if len(targets) == 1:
        return targets[0]      # 하나만 주문했을 때
    return targets             # 여러 개면 리스트로 병렬 시작

workflow.add_conditional_edges(START, start_router)

# 2) 버거: start → patty·bun 병렬 → fan-in → set → complete
workflow.add_edge('g_burger_start', 'g_patty')
workflow.add_edge('g_burger_start', 'g_bun')
workflow.add_edge(['g_patty', 'g_bun'], 'g_burger_set')   # 둘 다 끝나야 set
workflow.add_edge('g_burger_set', 'g_burger_complete')
workflow.add_edge('g_burger_complete', 'g_check_done')

# 3) 감자튀김: start → fry → set → complete
workflow.add_edge('g_fries_start', 'g_fry')
workflow.add_edge('g_fry', 'g_fries_set')
workflow.add_edge('g_fries_set', 'g_fries_complete')
workflow.add_edge('g_fries_complete', 'g_check_done')

# 4) 음료: 조리+완료 후 check
workflow.add_edge('g_beverage', 'g_check_done')

# 5) 완료 확인 후 종료
workflow.add_edge('g_check_done', END)

app = workflow.compile()

In [86]:
# 예시 1: 세 메뉴 전부
init_state = RestaurantState(menu_ordered=['burger', 'fries', 'beverage'])
result = app.invoke(init_state)
result


음료 준비 시작버거 준비 시작

감자튀김 준비 시작
음료 준비 완료
번 조리 시작
감자튀김 조리 시작
패티 조리 시작
번 조리 완료
감자튀김 조리 완료
패티 조리 완료
버거 세트 조립 시작
감자튀김 세트 준비 시작
감자튀김 세트 준비 완료
버거 세트 조립 완료
버거 완성
감자튀김 완성
주문하신 버거, 감자튀김, 음료 나왔습니다


{'menu_ordered': ['burger', 'fries', 'beverage'],
 'menu_complete': ['beverage', 'burger', 'fries'],
 'served': True}

In [87]:
# 예시 2: 버거만
init_state = RestaurantState(menu_ordered=['burger'])
result = app.invoke(init_state)
result


버거 준비 시작
번 조리 시작
패티 조리 시작
번 조리 완료
패티 조리 완료
버거 세트 조립 시작
버거 세트 조립 완료
버거 완성
주문하신 버거 나왔습니다


{'menu_ordered': ['burger'], 'menu_complete': ['burger'], 'served': True}

In [88]:
# 예시 3: 감자튀김 + 음료
init_state = RestaurantState(menu_ordered=['fries', 'beverage'])
result = app.invoke(init_state)
result

음료 준비 시작
감자튀김 준비 시작
음료 준비 완료
감자튀김 조리 시작
감자튀김 조리 완료
감자튀김 세트 준비 시작
감자튀김 세트 준비 완료
감자튀김 완성
주문하신 감자튀김, 음료 나왔습니다


{'menu_ordered': ['fries', 'beverage'],
 'menu_complete': ['beverage', 'fries'],
 'served': True}